# SIM 875s — Hole deformation characterisation (DEFC)

Plots `etaA` (area-based) and `etaC` (aspect-ratio-based) localization indices vs time for Step-1.

- `etaA` = `etaC_A` = `1 − std(A_normalised) / stdM`
- `etaC` = `etaC_R` = `1 − std(R_normalised) / stdM`

Labels show the number of elements in each mesh.

DEFC2 PKLs are generated by:
```bash
python -m A001_functions.compute_DEFC <SIM_NUMBER>
```
which requires `TP1.pkl` and `C.pkl` to exist for the simulation.

In [ ]:
import sys, os, pickle, json
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

RESULTS_DIR = '../I001_Results'
OBJ_DIR     = '../I001_Results/OBJ_files'
SIMS        = [875, 876, 877]


In [ ]:
def load_pkl(path):
    if not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)

def get_n_elements(mesh_file):
    """Read n_elements from the mesh JSON referenced in DEFC2."""
    if not mesh_file or not os.path.exists(mesh_file):
        return None
    with open(mesh_file) as f:
        m = json.load(f)
    return m.get('n_elements')

def step1_mask(t, displacement):
    """Return boolean mask selecting Step-1 frames (displacement non-zero)."""
    disp = np.array(displacement)
    # Step-1 starts where compression displacement first becomes non-trivial
    mask = np.abs(disp) > 1e-6
    # include the very first Step-1 frame
    if mask.any():
        first = np.argmax(mask)
        mask[first] = True
    return mask


In [ ]:
# ── Load DEFC2 + TP2 data ──────────────────────────────────────────────────────
data = {}

for sim in SIMS:
    defc2_path = os.path.join(RESULTS_DIR, f'DATA_PICK_{sim:03d}_DEFC2.pkl')
    tp2_path   = os.path.join(RESULTS_DIR, f'DATA_PICK_{sim:03d}_TP2.pkl')

    d2  = load_pkl(defc2_path)
    tp2 = load_pkl(tp2_path)

    if d2 is None and tp2 is None:
        print(f'SIM_{sim}: no DEFC2 or TP2 — skipping')
        continue

    # Use DEFC2 as time/displacement reference; fall back to TP2
    ref  = d2 if d2 is not None else tp2
    t    = np.array(ref['t'])
    disp = np.array(ref.get('displacement', np.zeros(len(t))))
    mask = step1_mask(t, disp)

    n_el  = get_n_elements((d2 or {}).get('source_file', ''))
    label = f'SIM_{sim}  ({n_el} elements)' if n_el else f'SIM_{sim}'

    entry = {'t': t[mask], 'disp': -disp[mask], 'label': label, 'n_el': n_el}

    if d2 is not None:
        entry['etaA'] = np.array(d2['etaC_A'])[mask]
        entry['etaC'] = np.array(d2['etaC_R'])[mask]

    if tp2 is not None:
        tp2_t    = np.array(tp2['t'])
        tp2_disp = disp if len(disp) == len(tp2_t) else np.zeros(len(tp2_t))
        tp2_mask = step1_mask(tp2_t, tp2_disp)
        entry['t_tp2']      = tp2_t[tp2_mask]
        entry['edi_mean']   = np.array(tp2['edi_mean'])[tp2_mask]
        entry['shear_mean'] = np.array(tp2['shear_mean'])[tp2_mask]
        entry['gle_mean']   = np.array(tp2['gle_mean'])[tp2_mask]
        entry['shear_mean_aw'] = np.array(tp2['shear_mean_aw'])[tp2_mask] if 'shear_mean_aw' in tp2 else None
        entry['gle_mean_aw']   = np.array(tp2['gle_mean_aw'])[tp2_mask]   if 'gle_mean_aw'   in tp2 else None
        entry['edi_mean_aw']   = np.array(tp2['edi_mean_aw'])[tp2_mask]   if 'edi_mean_aw'   in tp2 else None
        entry['eta_shear']     = np.array(tp2['eta_shear'])[tp2_mask]     if 'eta_shear'     in tp2 else None

    data[sim] = entry
    has = [k for k in ('etaA', 'etaC', 'edi_mean', 'shear_mean', 'gle_mean', 'edi_mean_aw', 'shear_mean_aw', 'gle_mean_aw', 'eta_shear') if k in entry]
    print(f'SIM_{sim}: {n_el} elements | {mask.sum()} Step-1 frames | fields: {has}')

print(f'\nLoaded {len(data)} simulations.')


## Plot 1 — `etaA` (area-based) vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]['n_el'] or 0):
    ax.plot(d['t'], d['etaA'], label=d['label'])

ax.set_xlabel('Time [s]  (Step-1)')
ax.set_ylabel(r'$\eta_A = 1 - \sigma(A^*)/\sigma_M$')
ax.set_title(r'Area-based localisation index $\eta_A$ — SIM 875s')
ax.legend(loc='best', fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 2 — `etaC` (aspect-ratio-based) vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]['n_el'] or 0):
    ax.plot(d['t'], d['etaC'], label=d['label'])

ax.set_xlabel('Time [s]  (Step-1)')
ax.set_ylabel(r'$\eta_C = 1 - \sigma(R^*)/\sigma_M$')
ax.set_title(r'Aspect-ratio-based localisation index $\eta_C$ — SIM 875s')
ax.legend(loc='best', fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Both on the same figure (optional)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for sim, d in sorted(data.items(), key=lambda x: x[1]['n_el'] or 0):
    axes[0].plot(d['t'], d['etaA'], label=d['label'])
    axes[1].plot(d['t'], d['etaC'], label=d['label'])

for ax, title, ylabel in zip(
    axes,
    [r'$\eta_A$ (area)', r'$\eta_C$ (aspect ratio)'],
    [r'$\eta_A$', r'$\eta_C$'],
):
    ax.set_xlabel('Time [s]  (Step-1)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc='best', fontsize=8)
    ax.grid(True)

fig.suptitle('Hole deformation localisation indices — SIM 875s', fontsize=13)
plt.tight_layout()
plt.show()


## Plot 3 — EDI mean vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]['n_el'] or 0):
    if 'edi_mean' not in d:
        continue
    ax.plot(d['t_tp2'], d['edi_mean'], label=d['label'])

ax.set_xlabel('Time [s]  (Step-1)')
ax.set_ylabel('EDI mean')
ax.set_title('EDI mean vs time — SIM 875s')
ax.legend(loc='best', fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 4 — Shear mean vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]['n_el'] or 0):
    if 'shear_mean' not in d:
        continue
    ax.plot(d['t_tp2'], d['shear_mean'], label=d['label'])

ax.set_xlabel('Time [s]  (Step-1)')
ax.set_ylabel('Shear mean')
ax.set_title('Shear mean vs time — SIM 875s')
ax.legend(loc='best', fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 5 — GLE mean vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]['n_el'] or 0):
    if 'gle_mean' not in d:
        continue
    ax.plot(d['t_tp2'], d['gle_mean'], label=d['label'])

ax.set_xlabel('Time [s]  (Step-1)')
ax.set_ylabel('GLE mean')
ax.set_title('GLE mean vs time — SIM 875s')
ax.legend(loc='best', fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 6 — EDI mean (area-weighted) vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]["n_el"] or 0):
    if d.get("edi_mean_aw") is None:
        continue
    ax.plot(d["t_tp2"], d["edi_mean_aw"], label=d["label"])

ax.set_xlabel("Time [s]  (Step-1)")
ax.set_ylabel("EDI mean (area-weighted)")
ax.set_title("EDI mean (area-weighted) vs time — SIM 875s")
ax.legend(loc="best", fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 7 — Shear mean (area-weighted) vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]["n_el"] or 0):
    if d.get("shear_mean_aw") is None:
        continue
    ax.plot(d["t_tp2"], d["shear_mean_aw"], label=d["label"])

ax.set_xlabel("Time [s]  (Step-1)")
ax.set_ylabel("Shear mean (area-weighted)")
ax.set_title("Shear mean (area-weighted) vs time — SIM 875s")
ax.legend(loc="best", fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 8 — GLE mean (area-weighted) vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]["n_el"] or 0):
    if d.get("gle_mean_aw") is None:
        continue
    ax.plot(d["t_tp2"], d["gle_mean_aw"], label=d["label"])

ax.set_xlabel("Time [s]  (Step-1)")
ax.set_ylabel("GLE mean (area-weighted)")
ax.set_title("GLE mean (area-weighted) vs time — SIM 875s")
ax.legend(loc="best", fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Plot 9 — $\eta_{\mathrm{shear}}$ (foam shear index) vs time — Step-1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for sim, d in sorted(data.items(), key=lambda x: x[1]["n_el"] or 0):
    if d.get("eta_shear") is None:
        continue
    ax.plot(d["t_tp2"], d["eta_shear"], label=d["label"])

ax.set_xlabel("Time [s]  (Step-1)")
ax.set_ylabel(r"$\eta_{\mathrm{shear}} = 1 - \sigma(S_e)/\sigma_i$")
ax.set_title(r"Foam shear index $\eta_{\mathrm{shear}}$ vs time — SIM 875s")
ax.legend(loc="best", fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()
